# Evaluation Metrics From Scratch

Before we can trust a model, we need to measure how good it is.
Different problems need different measuring sticks.

This notebook covers two groups:

| Group | Problem type | Example |
|-------|-------------|---------|
| **Regression metrics** | Predicting a number | Predict house price |
| **Classification metrics** | Predicting a category | Predict spam / not spam |

We implement every metric from scratch using only Python and pandas, then verify with sklearn.

---
# Part 1 — Regression Metrics

Regression metrics measure **how far off our predicted numbers are from the real numbers**.

We use a simple example: a model predicts exam scores.

In [ ]:
import pandas as pd
import math
import numpy as np
import matplotlib.pyplot as plt

y     = [10, 20, 30, 40]   # actual values
y_hat = [12, 18, 25, 42]   # model predictions

df = pd.DataFrame({'Actual (y)': y, 'Predicted (y_hat)': y_hat})
df['Error (y - y_hat)'] = df['Actual (y)'] - df['Predicted (y_hat)']
df

## MAE — Mean Absolute Error

**Question: On average, how many units was the model wrong?**

$$
MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|
$$

We take the absolute value so that positive and negative errors do not cancel each other out.

**How to read it:** If MAE = 2.75, the model is off by 2.75 units on average.
It does not tell you the direction, just the average size of the mistake.

In [ ]:
df['Abs Error'] = abs(df['Actual (y)'] - df['Predicted (y_hat)'])

mae = df['Abs Error'].sum() / len(df)

print(df.to_string(index=False))
print(f'\nMAE = {mae}')
print(f'The model is off by {mae} units on average.')

## MSE — Mean Squared Error

**Question: What is the average squared error?**

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

We square the errors instead of taking the absolute value. This has two effects:
1. Errors cannot cancel out (negatives become positive after squaring)
2. **Big errors get punished much more** — a mistake of 10 contributes 100 to MSE, while a mistake of 2 contributes only 4

**Downside:** The result is in squared units (e.g. squared dollars, squared kg) which is hard to interpret directly.

In [ ]:
df['Squared Error'] = (df['Actual (y)'] - df['Predicted (y_hat)']) ** 2

mse = df['Squared Error'].sum() / len(df)

print(df[['Actual (y)', 'Predicted (y_hat)', 'Abs Error', 'Squared Error']].to_string(index=False))
print(f'\nMSE = {mse}')
print('Units are squared, so hard to interpret directly.')

### Why high MSE?

Two common reasons:

**1. The model is too simple (underfitting)**
The model ignores the data and predicts the same value for everyone.

In [ ]:
y_actual = [10, 20, 30, 40]
y_dumb   = [25, 25, 25, 25]   # model always predicts 25

mse_dumb = sum((a - p)**2 for a, p in zip(y_actual, y_dumb)) / len(y_actual)
print(f'Dumb model (always predicts 25) MSE = {mse_dumb}')
print('High MSE because the model learned nothing.')

**2. One outlier explodes MSE**

Even if the model is great on most points, one very wrong prediction gets squared and dominates the score.

In [ ]:
y_actual  = [10, 20, 30, 200]   # last point is an outlier
y_pred    = [12, 22, 28,  50]   # model is close on the first 3

squared_errors = [(a - p)**2 for a, p in zip(y_actual, y_pred)]
mse_outlier = sum(squared_errors) / len(y_actual)

for i, (a, p, se) in enumerate(zip(y_actual, y_pred, squared_errors)):
    print(f'  Point {i+1}: actual={a:4d}, pred={p:4d}, squared error={se}')
print(f'\nMSE = {mse_outlier}  (one outlier made MSE explode)')

## RMSE — Root Mean Squared Error

**Question: Same as MSE but back in the original units.**

$$
RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
$$

Taking the square root brings the error back to the same units as y.
If y is in kilograms, RMSE is in kilograms — easy to interpret.

**How to read it:** RMSE = 3.0 means the model is roughly 3 units off on average,
but big errors are weighted more heavily than small ones.

In [ ]:
# Recalculate cleanly on the original small dataset
y     = [10, 20, 30, 40]
y_hat = [12, 18, 25, 42]

mse_clean  = sum((a - p)**2 for a, p in zip(y, y_hat)) / len(y)
rmse_clean = math.sqrt(mse_clean)
mae_clean  = sum(abs(a - p) for a, p in zip(y, y_hat)) / len(y)

print(f'MAE  = {mae_clean}')
print(f'MSE  = {mse_clean}  (hard to interpret — squared units)')
print(f'RMSE = {rmse_clean:.4f}  (same units as y, easier to interpret)')

## MAE vs MSE vs RMSE — When to Use Which

| Metric | Punishes big errors? | Units | Best when... |
|--------|---------------------|-------|--------------|
| MAE | No — all errors treated equally | Same as y | Outliers exist and you do not want them to dominate |
| MSE | Yes — squares them | Squared | You want to penalize large errors heavily |
| RMSE | Yes — but readable | Same as y | You want big-error penalty AND readable units |

**Quick rule:**
- RMSE >> MAE → your data has outliers or large occasional errors
- RMSE ≈ MAE → errors are roughly uniform across all points

In [ ]:
metrics = {'Metric': ['MAE', 'MSE', 'RMSE'],
           'Value':  [mae_clean, mse_clean, rmse_clean]}

plt.figure(figsize=(6, 4))
plt.bar(metrics['Metric'], metrics['Value'], color=['steelblue', 'tomato', 'seagreen'])
plt.title('MAE vs MSE vs RMSE on Our Dataset')
plt.ylabel('Error value')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

---
# Part 2 — Classification Metrics

Classification metrics measure **how well the model separates classes**.

Accuracy alone is not always enough. Imagine a model that predicts
"not fraud" for every transaction. If 99% of transactions are genuine,
it gets 99% accuracy — but catches zero frauds.

We need better metrics. We use a credit default dataset:
- **Actual = 1**: customer actually defaulted
- **Predicted = 1**: model predicted they would default

In [ ]:
data = {
    "Customer":          [1,  2,  3,  4,  5,  6,  7,  8,  9,  10],
    "Income":            ["50K","30K","45K","25K","70K","20K","80K","22K","40K","28K"],
    "Credit Score":      [720, 600, 650, 550, 780, 500, 800, 580, 670, 560],
    "Actual (Default)":  [0,   1,   0,   1,   0,   1,   0,   1,   0,   1],
    "Predicted":         [0,   1,   1,   1,   0,   0,   0,   1,   0,   0]
}

df = pd.DataFrame(data)
df

## TP, TN, FP, FN — The Four Outcomes

Every prediction falls into one of four boxes:

| | Predicted = 1 | Predicted = 0 |
|-|--------------|---------------|
| **Actual = 1** | True Positive (TP) | False Negative (FN) |
| **Actual = 0** | False Positive (FP) | True Negative (TN) |

**In plain English (using a COVID test as analogy):**

| Name | What happened | Risk |
|------|--------------|------|
| **TP** | Test says sick, you ARE sick | None — correct |
| **TN** | Test says fine, you ARE fine | None — correct |
| **FP** | Test says sick, you are NOT sick | Panic, unnecessary treatment — Type 1 Error |
| **FN** | Test says fine, you ARE sick | You spread disease — Type 2 Error (dangerous) |

In [ ]:
actual    = df['Actual (Default)'].tolist()
predicted = df['Predicted'].tolist()

TP = sum(1 for a, p in zip(actual, predicted) if a == 1 and p == 1)
TN = sum(1 for a, p in zip(actual, predicted) if a == 0 and p == 0)
FP = sum(1 for a, p in zip(actual, predicted) if a == 0 and p == 1)
FN = sum(1 for a, p in zip(actual, predicted) if a == 1 and p == 0)

print(f'True  Positives (TP) — predicted default,  actually defaulted:     {TP}')
print(f'True  Negatives (TN) — predicted no default, actually no default:   {TN}')
print(f'False Positives (FP) — predicted default,  actually did NOT default: {FP}')
print(f'False Negatives (FN) — predicted no default, actually DID default:   {FN}')

## Confusion Matrix

The confusion matrix is just a table that shows all four outcomes at once.
It gives you a complete picture of where the model is going right and where it is going wrong.

In [ ]:
conf_matrix = np.array([[TP, FN],
                         [FP, TN]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(conf_matrix, cmap='Blues')

labels = [['TP', 'FN'], ['FP', 'TN']]
row_labels = ['Actual = 1', 'Actual = 0']
col_labels = ['Predicted = 1', 'Predicted = 0']

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(col_labels)
ax.set_yticklabels(row_labels)

for i in range(2):
    for j in range(2):
        val = conf_matrix[i, j]
        ax.text(j, i, f'{labels[i][j]} = {val}', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if val > conf_matrix.max() / 2 else 'black')

ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## Accuracy

**Question: Out of all predictions, how many were correct?**

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

**When accuracy fails:** If 95% of emails are not spam, a model that always predicts
'not spam' gets 95% accuracy — but it is completely useless.
That is why we need precision and recall.

In [ ]:
accuracy = (TP + TN) / (TP + TN + FP + FN)
print(f'Accuracy = (TP + TN) / total = ({TP} + {TN}) / {TP+TN+FP+FN} = {accuracy:.2f}')
print(f'The model got {accuracy*100:.0f}% of predictions correct.')

## Precision

**Question: Of all the times the model predicted YES, how often was it actually right?**

$$
Precision = \frac{TP}{TP + FP}
$$

Think of it as: **when the model raises an alarm, how trustworthy is that alarm?**

- High precision → few false alarms
- Low precision → model cries wolf too often

**Use precision when false positives are costly.**
Example: email spam filter — you do not want real emails going to spam (FP).

In [ ]:
precision = TP / (TP + FP)
print(f'Precision = TP / (TP + FP) = {TP} / ({TP} + {FP}) = {precision:.2f}')
print(f'When the model predicted default, it was right {precision*100:.0f}% of the time.')

## Recall (Sensitivity)

**Question: Of all the actual YES cases, how many did the model catch?**

$$
Recall = \frac{TP}{TP + FN}
$$

Think of it as: **how good is the model at finding all the real positives?**

- High recall → model catches most real positives
- Low recall → model misses many real positives (dangerous false negatives)

**Use recall when false negatives are costly.**
Example: cancer screening — you do not want to miss a real cancer case (FN).

In [ ]:
recall = TP / (TP + FN)
print(f'Recall = TP / (TP + FN) = {TP} / ({TP} + {FN}) = {recall:.2f}')
print(f'The model caught {recall*100:.0f}% of actual defaults.')

## Precision vs Recall — The Trade-off

Precision and recall pull in opposite directions:

- If you make the model more aggressive (predict YES more often) → recall goes up, precision goes down
- If you make it more conservative (predict YES less often) → precision goes up, recall goes down

| Situation | Prioritise |
|-----------|------------|
| Missing a positive is dangerous (cancer, fraud) | **Recall** |
| A false alarm is costly (spam filter, legal action) | **Precision** |
| Both matter equally | **F1 Score** |

## F1 Score

**Question: Can we get one number that balances both precision and recall?**

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

This is the **harmonic mean** of precision and recall.
It is closer to the lower of the two values — so a model cannot get a high F1
by being great at one and terrible at the other.

F1 = 1.0 is perfect. F1 = 0.0 is worst.

In [ ]:
f1 = 2 * (precision * recall) / (precision + recall)
print(f'Precision = {precision:.2f}')
print(f'Recall    = {recall:.2f}')
print(f'F1 Score  = 2 * ({precision:.2f} * {recall:.2f}) / ({precision:.2f} + {recall:.2f}) = {f1:.2f}')

## All Metrics Together

In [ ]:
print('='*45)
print(f'  TP = {TP}   TN = {TN}   FP = {FP}   FN = {FN}')
print('='*45)
print(f'  Accuracy  = {accuracy:.2f}')
print(f'  Precision = {precision:.2f}')
print(f'  Recall    = {recall:.2f}')
print(f'  F1 Score  = {f1:.2f}')
print('='*45)

## Compare With sklearn

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, classification_report
)

y_true = df['Actual (Default)']
y_pred = df['Predicted']

print('sklearn results:')
print(f'  Accuracy  = {accuracy_score(y_true, y_pred):.2f}')
print(f'  Precision = {precision_score(y_true, y_pred):.2f}')
print(f'  Recall    = {recall_score(y_true, y_pred):.2f}')
print(f'  F1 Score  = {f1_score(y_true, y_pred):.2f}')
print()
print('Full report:')
print(classification_report(y_true, y_pred, target_names=['No Default', 'Default']))

## Summary

### Regression
| Metric | Formula | Use when |
|--------|---------|----------|
| MAE | mean(|y - y_hat|) | Outliers exist, want simple average error |
| MSE | mean((y - y_hat)^2) | Want to penalise big errors heavily |
| RMSE | sqrt(MSE) | Want big-error penalty in readable units |

### Classification
| Metric | Formula | Use when |
|--------|---------|----------|
| Accuracy | (TP+TN) / total | Classes are balanced |
| Precision | TP / (TP+FP) | False alarms are costly (spam filter) |
| Recall | TP / (TP+FN) | Missing positives is dangerous (cancer, fraud) |
| F1 Score | 2 * P*R / (P+R) | Both precision and recall matter |

**Key insight:** Accuracy looks great but hides problems on imbalanced datasets.
Always look at precision, recall, and F1 together to get the full picture.